In [1]:
from jobflow import Job, Flow
from jobflow_remote import submit_flow, set_run_config
from autoplex.auto.GenMLFF.jobs import RSS, evaluate_mlip_ensemble

/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/.env/lib/python3.11/site-packages/nequip/__init__.py:20: UserWarning: !! PyTorch version 2.2.1+cu121 found. Upstream issues in PyTorch versions 1.13.* and 2.* have been seen to cause unusual performance degredations on some CUDA systems that become worse over time; see https://github.com/mir-group/nequip/discussions/311. The best tested PyTorch version to use with CUDA devices is 1.11; while using other versions if you observe this problem, an unexpected lack of this problem, or other strange behavior, please post in the linked GitHub issue.
  warnings.warn(


In [2]:
#Define RSS test parameters
rss_test_params = {
    "tag": "Fe2O3", #Tag of systems. It can also be used for setting up elements and stoichiometry.
    "generated_struct_numbers": [10, 5], #Expected number of generated randomized unit cells for each run
    "buildcell_options": [ #Buildcell params for each buildcell run
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{6,8,10,12,14,16,18,20,22,24}'}, 
        {"NFORM": '1', "SYMMOPS": "1-4", "SLACK": 0.25, "OVERLAP" : 0.1, "NATOM" : '{7,9,11,13,15,17,19,21,23}'}
    ],
    "fragment_file": None, #Fragment(s) for random structures, e.g. molecules, to be placed indivudally intact.
    "remove_tmp_files": True, #Remove all temporary files raised by buildcell to save memory
    "num_processes": 1, #Number of processes to use for parallel computation
}

In [3]:
#Define parameters for the ensemble evaluator
mlip_ensemble_params = {
    "mlip_type": "MACE",
    "mlip_paths": [
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/3A/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/4A/small/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/4A/ultra_small/MACE_stagetwo.model",
        "/leonardo_scratch/fast/EUHPC_A04_113/Alberto/MLIP/MACE/Test-training/FeCOH/MP/cutoff-trials/5A/MACE_stagetwo.model",
        ],
    "mlip_errors": [151.3, 125.6, 153.7, 130.8],
    "mlip_kwargs": {"device" : "cuda"},
    "pre_trained_model": None,
    "pre_trained_kwargs": None,
}

In [4]:
#Define resources
parallel_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 32,
    "cpus_per_task": 1,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_cpu_resources = {
    "account": "IscrB_MLSilDia",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 32,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_gpu_resources = {
    "account": "IscrB_MLSilDia", 
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:1",
    "mem": "120000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
    }

In [5]:
#Define the RSS job
randomized_structures_paths = RSS(**rss_test_params)

#Define the ensemble evaluator job
evaluate_mlip_ensemble = evaluate_mlip_ensemble(**mlip_ensemble_params, structure_paths=randomized_structures_paths.output)

In [6]:
# Define the wrokflow
rss_ensemble_flow = Flow(jobs=[randomized_structures_paths, evaluate_mlip_ensemble], name="rss_ensemble_flow")

In [7]:
rss_ensemble_flow = set_run_config(
    rss_ensemble_flow, name_filter="RSS", worker="mlff_relax_local", exec_config="rss_config", resources=serial_cpu_resources
)

rss_ensemble_flow = set_run_config(
    rss_ensemble_flow, name_filter="evaluate_mlip_ensemble", worker="mlff_mace", resources=serial_gpu_resources
)

In [8]:
# Append RSSautoplex-flow to jf jobs
submit_flow(
    rss_ensemble_flow, worker="local_worker",
    resources={}, 
    project="GenMLFF",
)

2025-05-14 17:15:08,382 - INFO - Added flow (9b986262-4509-4fe8-b228-47b7aaa29634) with jobs: ('02d5c155-ac73-46cb-828b-97fd4ca7c1e0', '4fc46fb2-bb69-4492-98c2-ff736505d476')


['96', '97']